In [2]:
# install/update compatible HF libraries (run once; restart kernel if required)
%pip install -q --upgrade "diffusers>=0.19.0" "transformers>=4.30.0" "huggingface-hub>=0.15.0" accelerate safetensors

from diffusers import DiffusionPipeline
import torch




Note: you may need to restart the kernel to use updated packages.


ImportError: cannot import name 'DEFAULT_HF_PARALLEL_LOADING_WORKERS' from 'diffusers.utils.constants' (/opt/homebrew/Caskroom/miniforge/base/envs/tf_numpy2/lib/python3.11/site-packages/diffusers/utils/constants.py)

In [ ]:
# load both base & refiner
base = DiffusionPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0", torch_dtype=torch.float16, variant="fp16", use_safetensors=True
)
base.to("cuda")
refiner = DiffusionPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-refiner-1.0",
    text_encoder_2=base.text_encoder_2,
    vae=base.vae,
    torch_dtype=torch.float16,
    use_safetensors=True,
    variant="fp16",
)
refiner.to("cuda")

# Define how many steps and what % of steps to be run on each experts (80/20) here
n_steps = 40
high_noise_frac = 0.8

prompt = "A majestic lion jumping from a big stone at night"

# run both experts
image = base(
    prompt=prompt,
    num_inference_steps=n_steps,
    denoising_end=high_noise_frac,
    output_type="latent",
).images
image = refiner(
    prompt=prompt,
    num_inference_steps=n_steps,
    denoising_start=high_noise_frac,
    image=image,
).images[0]